# Jour 3 — CNN, classification d'images, transfer learning

Sujet : **chats vs chiens** (Microsoft Cats vs Dogs Dataset, Kaggle).

Trois itérations sur le même problème : CNN from scratch (TP1), augmentation + dropout (TP2), transfer learning MobileNetV2 (TP3).

## Phase 1.1 : Setup Colab + organisation du dataset

**Objectif** : configurer l'environnement Colab, télécharger le dataset via l'API Kaggle, organiser les dossiers en structure `train/val` par classe, et afficher quelques images pour vérifier.

Dataset choisi : [Microsoft Cats vs Dogs Dataset](https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset) — 25 000 images (12 500 chats, 12 500 chiens), livré en `PetImages/Cat` et `PetImages/Dog`.

In [ ]:
# === CONFIGURATION DU PROJET ===
# Seul endroit à changer si vous voulez switcher de sujet.
CLASS_A = "cat"
CLASS_B = "dog"

DATA_ROOT = "/content/data"

### Setup de l'API Kaggle

Nécessite un fichier `kaggle.json` (token d'authentification), téléchargeable depuis `kaggle.com/settings` > API > "Create New Token".

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

### Organisation des dossiers

Structure finale : `train/cat/`, `train/dog/`, `val/cat/`, `val/dog/` (split 80/20, seed=42).

Ce dataset précis contient quelques fichiers connus corrompus (0 octet ou JPEG invalide, ex. `Cat/666.jpg`, `Dog/11702.jpg`). On les filtre avec `PIL.Image.verify()` avant de les copier, sinon `image_dataset_from_directory` plantera en phase 1.2.

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


@tf.function
def _try_decode(img_bytes):
    # Force l'execution en mode graphe (via tf.function), qui est plus stricte que
    # l'execution eager sur certains formats (ex. BMP mal forme deguise en .jpg) et
    # reproduit exactement le mode d'execution utilise par image_dataset_from_directory
    # pendant l'entrainement.
    return tf.io.decode_image(img_bytes, channels=3, expand_animations=False)


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ni tf.io.decode_image() en
    # mode eager ne suffisent : ce dataset contient des fichiers que seul le decodeur en
    # mode graphe rejette (InvalidArgumentError, "Unknown image file format" ou erreurs
    # de dimension BMP). On teste donc chaque fichier via une tf.function.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            _try_decode(img_bytes)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)

In [ ]:
# Vérification : ce bloc doit tourner sans modification
for split in ['train', 'val']:
    for cls in [CLASS_A, CLASS_B]:
        path = os.path.join(DATA_ROOT, split, cls)
        print(f"{path} : {len(os.listdir(path))} images")

### Vérification visuelle

Grille 2x3 : 3 images de chaque classe, prises dans le train set.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_a = os.listdir(os.path.join(DATA_ROOT, "train", CLASS_A))[:3]
sample_b = os.listdir(os.path.join(DATA_ROOT, "train", CLASS_B))[:3]

plt.figure(figsize=(9, 6))
for i, fname in enumerate(sample_a):
    img = mpimg.imread(os.path.join(DATA_ROOT, "train", CLASS_A, fname))
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(CLASS_A)
    plt.axis("off")

for i, fname in enumerate(sample_b):
    img = mpimg.imread(os.path.join(DATA_ROOT, "train", CLASS_B, fname))
    plt.subplot(2, 3, i + 4)
    plt.imshow(img)
    plt.title(CLASS_B)
    plt.axis("off")

plt.tight_layout()
plt.savefig("sample_grid.png", dpi=100)
plt.show()

### Checklist avant de passer à la phase 1.2

- **Happy path** : au moins ~10 000 images en train par classe, ratio 80/20 respecté, grille 2x3 affichée sans erreur.
- **Edge case** : changer `seed` (0 vs 42) — le split change mais le ratio 80/20 reste identique à une unité près.
- **Adversarial** : `kaggle.json` mal placé ou permissions trop ouvertes → `OSError: Could not find kaggle.json` ou avertissement de permissions. Fix : `chmod 600 ~/.kaggle/kaggle.json`.